# ColorMeZO : la campagne LLM (SST-2, OPT-125M)

**Idée 24** de `mathisfrinzi/projet-recherche-mathis` · remplace le pilote (1 graine, 1 phrase, 15 pas, η non réglé).

| | pilote actuel | **cette campagne** |
|:---|:---|:---|
| tâche | perte LM d'une phrase fixe | **SST-2** (classification par prompt `… It was great/terrible`, style MeZO) |
| η | 1 valeur a priori | **réglé par méthode** sur un jeu de réglage disjoint du test |
| graines | 1 | **3 graines × méthodes (5 avec `MODE_FINALE`)** (même sous-ensemble et mêmes minibatchs par graine : comparaison appariée) |
| budget | 30 lectures | **2 000 lectures** (checkpoints dont 1 000) |
| mesure | perte d'une phrase | **perte et précision sur SST-2 validation (872 ex.)** |

## Question pré-enregistrée Q3
Sur une tâche réelle, quand la perte ne se lit qu'à travers un capteur `u = log10(g·ℓ)` quantifié 12 bits dont `ln g` dérive de σ_g = 0,05 **à chaque lecture**, ColorMeZO réduit-il la perte de validation plus que MeZO lisant le même capteur (η réglé pour chacun, 3 graines, 1 000–2 000 lectures) ?

## Prédictions (à relire et figer **avant** de lancer la campagne)
- **P6a, contrôle C1** : à capteur stable, MeZO ≈ oracle (écart de perte de validation ≤ 1 écart-type inter-graines).
- **P6b, Q3** : sous dérive, MeZO-capteur n'améliore pas (ou très peu) la perte de validation ; ColorMeZO la fait baisser nettement plus, dans le sens de l'étude 1.
- **P6c, le « prix 2× » du jouet ne se retrouve pas** : à capteur stable et η réglé, ColorMeZO ≈ MeZO, car la perte d'un LLM varie de moins de ~30 % sur la course et le η réglé absorbe le facteur 1/ℓ ; le ×2 du jouet venait d'une chute de ℓ sur deux décades.
- **Critère de réfutation** : si ColorMeZO-dérivant ne bat pas MeZO-dérivant sur au moins 2 graines sur 3 (à 1 000 et à 2 000 lectures), la transposition au LLM est réfutée à cette échelle ; on le consigne tel quel.

## Garde-fous identiques pour toutes les méthodes
ε = 1e-3 · perturbation gaussienne régénérée depuis la graine (aucune copie des poids) · restauration fusionnée à la mise à jour · pas non borné (la borne 0,1 de la campagne 1 écrêtait 80 % des pas) · fp32 sans TF32 (le bruit numérique doit rester très sous le signal) · poids de position non entraînés (comme le pilote).

## Mode d'emploi
Colab : *Exécution → Modifier le type d'exécution → GPU (T4)*, puis *Tout exécuter*. Les runs sont **sauvegardés un par un** dans `colormezo_llm_results.json`, et une archive est téléchargée après chaque méthode (réglage) et chaque graine (campagne finale). Après une déconnexion : déposer le dernier `colormezo_llm_results.json` dans le panneau *Fichiers*, puis relancer ; les runs déjà faits sont sautés.

Résultats et analyse : [`../rapport/5_llm.md`](../rapport/5_llm.md). Configuration par défaut = campagne 2.

## 1 · Configuration

In [ ]:
# ===================== CONFIG =====================
PRESET = "full"   # "full" (~1 h sur T4, estimation non mesurée) | "quick" (~20 min) | "smoke" (test du code : modèle jouet, CPU, secondes)

# Sur Colab, pour survivre à une déconnexion : monter Drive et pointer ici
RESULTS_PATH = "colormezo_llm_results.json"

_COMMON = dict(model="facebook/opt-125m", batch=16, n_train=1000, n_dev_tune=500, eval_bs=128)
PRESETS = {
    "full":  dict(_COMMON, final_reads=2000, tune_reads=1000, tune_seeds=[101, 102],
                  eta_grid=[1e-8, 3e-8, 1e-7, 3e-7, 1e-6, 3e-6, 1e-5, 3e-5, 1e-4], seeds=[0, 1, 2]),
    # campagne 2 : grille de 1e-8 à 3e-6 ; prolongée ensuite à 1e-4 (les runs déjà faits sont réutilisés)
    "quick": dict(_COMMON, final_reads=1000, tune_reads=600, tune_seeds=[101],
                  eta_grid=[1e-7, 3e-7, 1e-6, 3e-6, 1e-5], seeds=[0, 1, 2]),
    "smoke": dict(model="toy", batch=8, n_train=64, n_dev_tune=32, eval_bs=16,
                  final_reads=24, tune_reads=12, tune_seeds=[101],
                  eta_grid=[1e-6, 1e-4], seeds=[0, 1]),
}

# ---- Run "finale" (sans réglage) : η déjà choisis sur la précision de réglage (campagne 3), plus de graines de test.
# Mettre MODE_FINALE = True pour sauter le réglage (~50 min sur T4 pour 5 méthodes × 5 graines × 2 000 lectures).
MODE_FINALE = False
ETA_FIXE = {"oracle": 1e-6, "mezo_stable": 1e-6, "color_stable": 3e-6,
            "mezo_drift": 1e-6, "color_drift": 3e-6}          # η* selon la précision de réglage (rapport 5.7)
SEEDS_TEST = [0, 1, 2, 3, 4]   # utilisé seulement si MODE_FINALE ; les graines 0 à 2 sont celles des campagnes précédentes
CFG = PRESETS[PRESET]
if MODE_FINALE:
    CFG = dict(CFG, seeds=SEEDS_TEST)

# Capteur et optimiseur : identiques pour toutes les méthodes
EPS = 1e-3            # amplitude de la perturbation SPSA
SIGMA_G = 0.05        # dérive de ln g par lecture (le régime de l'idée 24)
BITS = 12             # quantification de u = log10(g·ℓ)
STEP_NORM_MAX = None  # campagne 1 : 0.1 -> 80 % des pas écrêtés, test non discriminant ; campagne 2 : None
TELECHARGER = True    # Colab : télécharge une archive des résultats après chaque méthode / graine
CRITERE_REGLAGE = "perte"   # "perte" (prévu, celui qui a tourné) ou "precision" : au-delà de 3e-6, la perte choisit
                            # des η qui atteignent un plateau sans polarité (précision ~0,5), voir rapport/5_llm.md §5.7
MAX_SENT_TOK = 56     # troncature de la phrase (le gabarit " It was" est toujours conservé)

# (mode de lecture, capteur dérivant ?)   mode ∈ oracle | lin | log | geo
CONFIGS = {
    "oracle":       ("oracle", None),
    "mezo_stable":  ("lin", False),
    "color_stable": ("log", False),
    "mezo_drift":   ("lin", True),
    "color_drift":  ("log", True),
    "geo_stable":   ("geo", False),   # re-étalonnage de l'étude 3 (optionnel)
    "geo_drift":    ("geo", True),
}
METHODS = ["oracle", "mezo_stable", "color_stable", "mezo_drift", "color_drift"]
# METHODS += ["geo_stable", "geo_drift"]   # décommenter pour ajouter la frontière de l'étude 3

LABEL = {
    "oracle": "MeZO oracle (perte exacte)",
    "mezo_stable": "MeZO capteur stable (C1)",
    "color_stable": "ColorMeZO capteur stable",
    "mezo_drift": "MeZO capteur dérivant",
    "color_drift": "ColorMeZO capteur dérivant",
    "geo_stable": "ColorMeZO-geo capteur stable",
    "geo_drift": "ColorMeZO-geo capteur dérivant",
}

# checkpoints (lectures), toujours pairs : 2 lectures par pas
CKPTS = sorted({0} | {2 * int(round(CFG["final_reads"] * k / 16)) for k in range(1, 9)})
MID_READS, FINAL_READS = CKPTS[len(CKPTS) // 2], CFG["final_reads"]
print(f"preset={PRESET} · méthodes={METHODS}\nη grille={CFG['eta_grid']} · graines test={CFG['seeds']} · "
      f"réglage: {CFG['tune_seeds']} à {CFG['tune_reads']} lectures\ncheckpoints={CKPTS}")

In [ ]:
import importlib.util, subprocess, sys
_need = [m for m in ("transformers", "pandas", "matplotlib") if importlib.util.find_spec(m) is None]
if CFG["model"] != "toy" and importlib.util.find_spec("datasets") is None:
    _need.append("datasets")
if _need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_need])

import os, json, math, time
import numpy as np
import torch
import torch.nn.functional as Fnn

dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_grad_enabled(False)
torch.backends.cuda.matmul.allow_tf32 = False   # fp32 strict : le bruit numérique doit rester ≪ signal SPSA
torch.backends.cudnn.allow_tf32 = False
print("device:", dev, "·", torch.cuda.get_device_name(0) if dev == "cuda" else "(CPU)")
if dev == "cpu" and PRESET != "smoke":
    print("Pas de GPU : sur CPU cette campagne prendrait des dizaines d'heures. "
          "Colab → Exécution → Modifier le type d'exécution → GPU.")

## 2 · Données SST-2 (prompt « *phrase* It was great/terrible »)

In [ ]:
rng_data = np.random.default_rng(999)

if CFG["model"] == "toy":                       # --- mode smoke : données synthétiques, aucun téléchargement
    LW, BOS, PAD = [5, 6], 2, 1                 # "mots-étiquettes" jouets
    def _toy(n):
        return [([int(x) for x in rng_data.integers(10, 100, int(rng_data.integers(5, 12)))] + [7, 8], i % 2)
                for i in range(n)]
    ENC = lambda ids: [BOS] + ids
    POOL_ALL, VAL_RAW = _toy(400), _toy(48)
else:                                           # --- vraie campagne
    from transformers import AutoTokenizer
    from datasets import load_dataset
    tok = AutoTokenizer.from_pretrained(CFG["model"])
    ds = load_dataset("nyu-mll/glue", "sst2")
    BOS, PAD = tok.bos_token_id, tok.pad_token_id
    TEMPLATE = tok(" It was", add_special_tokens=False)["input_ids"]
    LW = [tok(w, add_special_tokens=False)["input_ids"] for w in (" terrible", " great")]
    assert all(len(x) == 1 for x in LW), "les mots-étiquettes doivent être des tokens uniques"
    LW = [x[0] for x in LW]
    ENC = lambda sent: [BOS] + tok(sent.strip(), add_special_tokens=False)["input_ids"][:MAX_SENT_TOK] + TEMPLATE
    POOL_ALL = list(zip(ds["train"]["sentence"], ds["train"]["label"]))
    VAL_RAW = list(zip(ds["validation"]["sentence"], ds["validation"]["label"]))

def build(raw):
    return [(ENC(p), int(l)) for p, l in raw]

# jeu de réglage de η : tiré du train, équilibré, DISJOINT de tous les sous-ensembles d'entraînement
_lab = np.array([l for _, l in POOL_ALL])
_dev_idx = np.concatenate([rng_data.choice(np.where(_lab == c)[0], CFG["n_dev_tune"] // 2, replace=False) for c in (0, 1)])
_dev_set = set(int(i) for i in _dev_idx)
POOL = [(p, l) for i, (p, l) in enumerate(POOL_ALL) if i not in _dev_set]
DEV_TUNE = build([POOL_ALL[int(i)] for i in _dev_idx])
VAL = build(VAL_RAW)                              # = « test » final (SST-2 validation)

_subset_cache = {}
def train_subset(seed):
    """n_train exemples équilibrés, fixés par la graine (identiques pour toutes les méthodes)."""
    if seed not in _subset_cache:
        r = np.random.default_rng(seed)
        pl = np.array([l for _, l in POOL])
        sel = np.concatenate([r.choice(np.where(pl == c)[0], CFG["n_train"] // 2, replace=False) for c in (0, 1)])
        r.shuffle(sel)
        _subset_cache[seed] = build([POOL[int(i)] for i in sel])
    return _subset_cache[seed]

def collate(exs):
    T = max(len(x) for x, _ in exs)
    ids = torch.full((len(exs), T), PAD, dtype=torch.long)
    mask = torch.zeros((len(exs), T), dtype=torch.long)
    for i, (x, _) in enumerate(exs):
        ids[i, :len(x)] = torch.tensor(x)
        mask[i, :len(x)] = 1
    lab = torch.tensor([l for _, l in exs])
    return dict(ids=ids.to(dev), mask=mask.to(dev),
                last=torch.tensor([len(x) - 1 for x, _ in exs]).to(dev),
                y=torch.tensor([LW[int(l)] for l in lab]).to(dev), lab=lab.to(dev))

print(f"train (pool) {len(POOL)} · réglage {len(DEV_TUNE)} · validation {len(VAL)} · "
      f"sous-ensemble d'entraînement par graine : {CFG['n_train']} (équilibré)")
print("exemple :", train_subset(0)[0][0][:12], "…", "· mots-étiquettes :", LW)

## 3 · Modèle, perte, perturbation en place

In [ ]:
from transformers import AutoModelForCausalLM, OPTConfig, OPTForCausalLM

if CFG["model"] == "toy":
    torch.manual_seed(0)
    model = OPTForCausalLM(OPTConfig(vocab_size=100, hidden_size=32, ffn_dim=64, num_hidden_layers=2,
                                     num_attention_heads=2, max_position_embeddings=128,
                                     word_embed_proj_dim=32, pad_token_id=PAD, bos_token_id=BOS))
else:
    model = AutoModelForCausalLM.from_pretrained(CFG["model"], torch_dtype=torch.float32)
model = model.to(dev).eval()

params = [p for n, p in model.named_parameters() if "embed_positions" not in n]
D = sum(p.numel() for p in params)
init_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
print(f"{D/1e6:.1f} M paramètres entraînés")

def last_logits(b):
    """Logits (full vocab) à la dernière position réelle de chaque exemple."""
    h = model.model.decoder(input_ids=b["ids"], attention_mask=b["mask"], use_cache=False).last_hidden_state
    h = h[torch.arange(h.shape[0], device=h.device), b["last"]]
    return model.lm_head(h)

def lm_loss(b):
    """Perte vraie du minibatch : CE plein vocabulaire du mot-étiquette (comme MeZO)."""
    return Fnn.cross_entropy(last_logits(b), b["y"]).item()

def evaluate(exs):
    """(perte CE, précision restreinte aux 2 mots-étiquettes) sur un jeu complet."""
    tot, ok = 0.0, 0
    for i in range(0, len(exs), CFG["eval_bs"]):
        b = collate(exs[i:i + CFG["eval_bs"]])
        lg = last_logits(b)
        tot += Fnn.cross_entropy(lg, b["y"], reduction="sum").item()
        ok += (lg[:, LW].argmax(1) == b["lab"]).sum().item()
    return tot / len(exs), ok / len(exs)

gen = torch.Generator(device=dev)
def perturb(c, zseed):
    """θ += c·z, z régénéré depuis zseed (aucune copie des poids)."""
    gen.manual_seed(zseed)
    for p in params:
        p.data.add_(torch.randn(p.shape, device=p.device, dtype=p.dtype, generator=gen), alpha=c)

l0, a0 = evaluate(VAL)
print(f"avant entraînement, validation : perte {l0:.4f} · précision {a0:.3f}")

## 4 · Capteur couleur et estimateurs de gradient

In [ ]:
LN10 = math.log(10.0)

class Sensor:
    """u = log10(g·ℓ) quantifié ; ln g dérive de N(0, σ_g²) à CHAQUE lecture (marche aléatoire)."""
    def __init__(self, seed, drift):
        self.rng, self.drift, self.ln_g, self.ln_g_max = np.random.default_rng(seed + 777), drift, 0.0, 0.0
    def read(self, loss):
        if self.drift:
            self.ln_g += self.rng.normal(0, SIGMA_G)
            self.ln_g_max = max(self.ln_g_max, abs(self.ln_g))
        u = math.log10(math.exp(self.ln_g) * max(loss, 1e-300))
        return float(np.round(u * 2 ** BITS) / 2 ** BITS)

def estimate_g(mode, lp, lm, sensor):
    """Pente SPSA à partir des deux lectures (lp en θ+εz, lm en θ−εz)."""
    if mode == "oracle":                       # perte exacte
        return (lp - lm) / (2 * EPS)
    up, um = sensor.read(lp), sensor.read(lm)
    if mode == "lin":                          # MeZO : valeurs reconstruites 10^u (décennie connue)
        delta = 10.0 ** up - 10.0 ** um
    else:                                      # ColorMeZO : différence de ln ℓ, dépliée à ±0,5 décennie
        d = up - um
        delta = LN10 * (d - float(np.round(d)))
        if mode == "geo":                      # re-étalonnage : × moyenne géométrique des deux reconstructions
            delta *= math.sqrt(10.0 ** up * 10.0 ** um)
    return delta / (2 * EPS)

def run_once(cfg, eta, seed, reads, eval_exs, ckpts):
    mode, drift = CONFIGS[cfg]
    model.load_state_dict(init_state)          # même point de départ pour tous les runs
    tr = train_subset(seed)
    rng = np.random.default_rng(seed)          # mêmes minibatchs pour toutes les méthodes (appariement)
    sensor = None if mode == "oracle" else Sensor(seed, drift)
    clip = STEP_NORM_MAX / math.sqrt(D) if STEP_NORM_MAX else float("inf")   # borne sur |η·g|
    curve, n_clip, n_bad, gabs = [], 0, 0, []
    if 0 in ckpts:
        curve.append((0, *evaluate(eval_exs)))
    t0 = time.time()
    for t in range(reads // 2):
        b = collate([tr[i] for i in rng.choice(len(tr), CFG["batch"], replace=False)])
        zs = seed * 1_000_003 + t
        perturb(+EPS, zs);     lp = lm_loss(b)       # θ + εz
        perturb(-2 * EPS, zs); lm = lm_loss(b)       # θ − εz
        g = estimate_g(mode, lp, lm, sensor)
        c = eta * g
        if not math.isfinite(c):
            c, n_bad = 0.0, n_bad + 1
        elif abs(c) > clip:
            c, n_clip = math.copysign(clip, c), n_clip + 1
        gabs.append(abs(c / eta) if math.isfinite(c) else 0.0)
        perturb(EPS - c, zs)                          # restauration (+ε) fusionnée au pas (−c) : θ − c·z
        done = 2 * (t + 1)
        if done in ckpts:
            curve.append((done, *evaluate(eval_exs)))
    return dict(cfg=cfg, eta=eta, seed=seed, reads=reads, curve=curve,
                final_loss=curve[-1][1] if np.isfinite(curve[-1][1]) else float("inf"),
                clipped_frac=n_clip / max(1, reads // 2), nonfinite=n_bad,
                ln_g_max=getattr(sensor, "ln_g_max", 0.0), seconds=round(time.time() - t0, 1))

# --- sauvegarde run par run (reprise après déconnexion)
RESULTS = json.load(open(RESULTS_PATH)) if os.path.exists(RESULTS_PATH) else {}
def save_results():
    tmp = RESULTS_PATH + ".tmp"
    json.dump(RESULTS, open(tmp, "w"))
    os.replace(tmp, RESULTS_PATH)

def sauver():
    """Archive horodatée des résultats, téléchargée par le navigateur (sans Drive)."""
    if not TELECHARGER:
        return
    import zipfile
    fichiers = [f for f in (RESULTS_PATH, "colormezo_llm_sst2.png", "colormezo_llm_sst2.svg",
                            "readme_bloc_campagne_llm.md") if os.path.exists(f)]
    nom = f"colormezo_{time.strftime('%H%M%S')}.zip"
    with zipfile.ZipFile(nom, "w", zipfile.ZIP_DEFLATED) as z:
        for f in fichiers:
            z.write(f)
    try:
        from google.colab import files
        files.download(nom)
    except ImportError:
        pass

def cached(phase, cfg, eta, seed, reads, eval_exs, ckpts):
    key = f"{PRESET}|{phase}|{cfg}|{eta:.0e}|{seed}|{reads}"
    if key not in RESULTS:
        RESULTS[key] = run_once(cfg, eta, seed, reads, eval_exs, ckpts)
        save_results()
        r = RESULTS[key]
        print(f"[{phase}] {cfg:<12} η={eta:.0e} graine={seed} → perte {r['final_loss']:.4f} "
              f"({r['seconds']:.0f}s, écrêtés {r['clipped_frac']:.0%})", flush=True)
    return RESULTS[key]

# --- vérification rapide du code sur un mini-run (quelques pas)
_t = time.time(); _r = run_once("color_drift", 1e-6, 0, 4, DEV_TUNE[:CFG["eval_bs"]], {4})
print(f"mini-run de contrôle OK · {(time.time()-_t)/2:.2f} s/pas (2 lectures + 1 évaluation incluse)")

## 5 · Diagnostic (≈ 1 min) : le signal SPSA est-il lisible à travers le capteur ?
Compare, à l'initialisation, l'amplitude du signal `δ ln ℓ` (une paire ±εz) au bruit de quantification et au bruit de dérive. Si la quantification domine, augmenter `EPS`.

In [ ]:
model.load_state_dict(init_state)
b = collate(train_subset(0)[:CFG["batch"]])
ell0 = lm_loss(b)
dl = []
for s in range(40):
    perturb(+EPS, 5000 + s); lp = lm_loss(b)
    perturb(-2 * EPS, 5000 + s); lm = lm_loss(b)
    perturb(+EPS, 5000 + s)
    dl.append(lp - lm)
model.load_state_dict(init_state)

sig = float(np.std(dl)) / ell0                                   # écart-type de δ ln ℓ (signal)
q = LN10 * 2 ** -BITS / math.sqrt(12) * math.sqrt(2)             # bruit de quantification sur la paire
dr = SIGMA_G * math.sqrt(2)                                      # bruit de dérive sur la paire
print(f"perte initiale du minibatch : ℓ = {ell0:.3f}")
print(f"signal      std(δ ln ℓ) = {sig:.2e}")
print(f"quantif.    bruit       = {q:.2e}   (signal/bruit = {sig/q:.1f})")
print(f"dérive      bruit       = {dr:.2e}   (signal/bruit = {sig/dr:.2f}) ← par paire ; SPSA moyenne sur les pas")
if sig / q < 3:
    print("la quantification domine : essayer EPS plus grand (ex. 3e-3).")
else:
    print("le signal est bien au-dessus de la quantification 12 bits.")
if sig / dr < 1:
    print("sous dérive, une paire isolée est dominée par le bruit : c'est un vrai test de résistance, "
          "pas une formalité (les deux méthodes en souffrent ; MeZO en plus par l'échelle g qui se promène).")

## 6 · Réglage de η par méthode
Critère prévu (`CRITERE_REGLAGE = "perte"`) : **perte CE moyenne sur le jeu de réglage** (500 ex. du train, jamais utilisés en entraînement ni en test) à la fin de `tune_reads`, moyenne sur les graines de réglage (distinctes des graines de test). Un run divergent compte comme +∞. Variante `"precision"` : meilleure précision sur le même jeu. Les deux tables sont affichées.

In [ ]:
import pandas as pd
TUNE_TABLE, TUNE_ACC, BEST_ETA = {}, {}, {}
t_start = time.time()
if MODE_FINALE:
    BEST_ETA = {c: ETA_FIXE[c] for c in METHODS}
    print("MODE_FINALE : réglage sauté, η imposés par ETA_FIXE :")
    for cfg in METHODS:
        print(f"η* {LABEL[cfg]:<34} = {BEST_ETA[cfg]:.0e}")
else:
    for cfg in METHODS:
        row, row_acc = {}, {}
        for eta in CFG["eta_grid"]:
            rs = [cached("tune", cfg, eta, s, CFG["tune_reads"], DEV_TUNE, {CFG["tune_reads"]}) for s in CFG["tune_seeds"]]
            row[eta] = float(np.mean([r["final_loss"] for r in rs]))
            accs = [r["curve"][-1][2] for r in rs]
            row_acc[eta] = float(np.mean(accs)) if all(np.isfinite(accs)) and np.isfinite(row[eta]) else 0.0
        TUNE_TABLE[cfg], TUNE_ACC[cfg] = row, row_acc
        BEST_ETA[cfg] = (min(row, key=row.get) if CRITERE_REGLAGE == "perte" else max(row_acc, key=row_acc.get))
        sauver()
    print(f"\nréglage terminé en {(time.time()-t_start)/60:.1f} min")

    tt = pd.DataFrame(TUNE_TABLE).T
    tt.columns = [f"{e:.0e}" for e in tt.columns]
    tt.index = [LABEL[c] for c in tt.index]
    print(f"\nperte de réglage (initiale ≈ {evaluate(DEV_TUNE)[0]:.4f}), par η :")
    display(tt.round(4))
    ta = pd.DataFrame(TUNE_ACC).T
    ta.columns = [f"{e:.0e}" for e in ta.columns]
    ta.index = [LABEL[c] for c in ta.index]
    print("précision de réglage, par η :")
    display(ta.round(3))
    print(f"critère retenu : {CRITERE_REGLAGE}")
    for cfg in METHODS:
        g = CFG["eta_grid"]
        flag = "  au bord de la grille : étendre" if BEST_ETA[cfg] in (g[0], g[-1]) else ""
        print(f"η* {LABEL[cfg]:<34} = {BEST_ETA[cfg]:.0e}{flag}")

## 7 · Campagne finale : 3 graines × méthodes, évaluation sur SST-2 validation

In [ ]:
t_start = time.time()
for seed in CFG["seeds"]:                       # graine en boucle externe : résultats partiels exploitables si interruption
    for cfg in METHODS:
        cached("final", cfg, BEST_ETA[cfg], seed, FINAL_READS, VAL, set(CKPTS))
    sauver()
print(f"\ncampagne terminée en {(time.time()-t_start)/60:.1f} min")

## 8 · Résultats

In [ ]:
import matplotlib.pyplot as plt

FINAL = {}
for k, v in RESULTS.items():
    p = k.split("|")
    if p[0] == PRESET and p[1] == "final" and int(p[5]) == FINAL_READS and p[2] in METHODS \
            and float(p[3]) == BEST_ETA[p[2]]:
        FINAL.setdefault(p[2], {})[int(p[4])] = v

def at(cfg, seed, reads):
    for r, l, a in FINAL[cfg][seed]["curve"]:
        if r == reads:
            return l, a
    return float("nan"), float("nan")

def ms(vals):
    vals = np.array(vals, float)
    return f"{np.nanmean(vals):.4f} ± {np.nanstd(vals):.4f}"

rows = []
for cfg in METHODS:
    if cfg not in FINAL: continue
    S = sorted(FINAL[cfg])
    l_init = [at(cfg, s, 0)[0] for s in S]
    rows.append({
        "méthode": LABEL[cfg], "η*": f"{BEST_ETA[cfg]:.0e}", "n graines": len(S),
        "perte init": f"{np.mean(l_init):.4f}",
        f"perte @{MID_READS}": ms([at(cfg, s, MID_READS)[0] for s in S]),
        f"perte @{FINAL_READS}": ms([at(cfg, s, FINAL_READS)[0] for s in S]),
        "Δ perte": f"{np.mean([at(cfg, s, FINAL_READS)[0] - at(cfg, s, 0)[0] for s in S]):+.4f}",
        "préc. init": f"{np.mean([at(cfg, s, 0)[1] for s in S]):.3f}",
        f"préc. @{FINAL_READS}": f"{np.mean([at(cfg, s, FINAL_READS)[1] for s in S]):.3f} ± {np.std([at(cfg, s, FINAL_READS)[1] for s in S]):.3f}",
        "pas écrêtés": f"{np.mean([FINAL[cfg][s]['clipped_frac'] for s in S]):.0%}",
        "max |ln g|": f"{np.mean([FINAL[cfg][s]['ln_g_max'] for s in S]):.2f}",
    })
df = pd.DataFrame(rows).set_index("méthode")
print(f"SST-2 validation ({len(VAL)} ex.) · moyenne ± écart-type sur {len(CFG['seeds'])} graines · perte = CE plein vocabulaire du mot-étiquette")
display(df)

# ---- courbes
STYLE = {"oracle": ("black", "-"), "mezo_stable": ("tab:blue", "--"), "color_stable": ("tab:orange", "--"),
         "mezo_drift": ("tab:blue", "-"), "color_drift": ("tab:orange", "-"),
         "geo_stable": ("tab:green", "--"), "geo_drift": ("tab:green", "-")}
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
for cfg in METHODS:
    if cfg not in FINAL: continue
    S = sorted(FINAL[cfg])
    xs = [r for r, _, _ in FINAL[cfg][S[0]]["curve"]]
    for j, name in enumerate(("perte de validation", "précision de validation")):
        Y = np.array([[c[1 + j] for c in FINAL[cfg][s]["curve"]] for s in S])
        m, sd = Y.mean(0), Y.std(0)
        col, ls = STYLE[cfg]
        ax[j].plot(xs, m, color=col, ls=ls, lw=2, label=LABEL[cfg])
        ax[j].fill_between(xs, m - sd, m + sd, color=col, alpha=0.12)
        ax[j].set_xlabel("lectures du capteur"); ax[j].set_ylabel(name); ax[j].grid(alpha=0.3)
ax[0].legend(fontsize=8)
fig.suptitle(f"ColorMeZO sur SST-2 (OPT-125M), σ_g = {SIGMA_G}/lecture, {len(CFG['seeds'])} graines, bande = ±1 écart-type")
fig.tight_layout()
fig.savefig("colormezo_llm_sst2.png", dpi=150); fig.savefig("colormezo_llm_sst2.svg")
plt.show()

### Lecture des prédictions (calculée à partir des résultats, sans interprétation ajoutée)

In [ ]:
def paired(a, b, reads):
    S = sorted(set(FINAL.get(a, {})) & set(FINAL.get(b, {})))
    da = [at(a, s, reads)[0] for s in S]; db = [at(b, s, reads)[0] for s in S]
    return S, da, db

print("Q3 : ColorMeZO vs MeZO sous dérive (perte de validation, plus bas = mieux)")
wins_total = {}
for reads in (MID_READS, FINAL_READS):
    S, c, m = paired("color_drift", "mezo_drift", reads)
    if not S: continue
    wins = sum(x < y for x, y in zip(c, m)); wins_total[reads] = wins
    print(f"@{reads:>5} lectures : ColorMeZO mieux sur {wins}/{len(S)} graines · "
          f"moyennes {np.mean(c):.4f} vs {np.mean(m):.4f} (Δ = {np.mean(c) - np.mean(m):+.4f}) · "
          f"par graine : {[round(x - y, 4) for x, y in zip(c, m)]}")
init = np.mean([at("mezo_drift", s, 0)[0] for s in FINAL.get("mezo_drift", {})] or [float("nan")])
for cfg in ("mezo_drift", "color_drift"):
    if cfg in FINAL:
        d = [at(cfg, s, FINAL_READS)[0] - at(cfg, s, 0)[0] for s in sorted(FINAL[cfg])]
        print(f"   {LABEL[cfg]:<28} Δ perte vs init : {np.mean(d):+.4f}  (par graine {[round(x, 4) for x in d]})")
if len(wins_total) == 2:
    refut = all(w < 2 for w in wins_total.values())
    print("→ critère de réfutation (ColorMeZO ne gagne pas ≥ 2/3 aux deux budgets) :", "ATTEINT : transposition réfutée à cette échelle" if refut else "non atteint")

print("\nP6a (C1) : MeZO capteur stable vs oracle")
S, a, o = paired("mezo_stable", "oracle", FINAL_READS)
if S:
    sd_or = np.std([at("oracle", s, FINAL_READS)[0] for s in S])
    print(f"écart moyen {np.mean(a) - np.mean(o):+.4f} · écart-type inter-graines (oracle) {sd_or:.4f} → ",
          "≤ 1 écart-type : P6a tenue" if abs(np.mean(a) - np.mean(o)) <= sd_or else "> 1 écart-type : P6a non tenue")

print("\nP6c : ColorMeZO vs MeZO à capteur stable (η réglé)")
S, c, m = paired("color_stable", "mezo_stable", FINAL_READS)
if S:
    print(f"écart moyen {np.mean(c) - np.mean(m):+.4f} · par graine {[round(x - y, 4) for x, y in zip(c, m)]}")

print(f"\n(n = {len(CFG['seeds'])} graines : indicatif, aucun test statistique formel, comme le reste du projet.)")

## 9 · Bloc prêt à coller dans le README de l'idée 24

In [ ]:
init_l = np.mean([at(METHODS[0], s, 0)[0] for s in FINAL[METHODS[0]]])
init_a = np.mean([at(METHODS[0], s, 0)[1] for s in FINAL[METHODS[0]]])
lines = [
    f"## Campagne LLM réelle : SST-2, OPT-125M ({time.strftime('%Y-%m-%d')})",
    "",
    f"Tâche : SST-2 par prompt (`… It was great/terrible`), {CFG['n_train']} exemples d'entraînement équilibrés par graine, "
    f"minibatch {CFG['batch']}, évaluation sur SST-2 validation ({len(VAL)} ex.). Capteur `u = log10(g·ℓ)` {BITS} bits, "
    f"σ_g = {SIGMA_G}/lecture. η réglé par méthode ({len(CFG['tune_seeds'])} graine(s) de réglage, {CFG['tune_reads']} lectures, jeu de réglage disjoint). "
    f"Budget {FINAL_READS} lectures, {len(CFG['seeds'])} graines, ε = {EPS:g}, pas borné à {STEP_NORM_MAX}.",
    "",
    f"Avant entraînement : perte {init_l:.4f}, précision {init_a:.3f}.",
    "",
    f"| méthode | η | perte @{MID_READS} | perte @{FINAL_READS} | précision @{FINAL_READS} |",
    "|:---|:---:|:---:|:---:|:---:|",
]
for cfg in METHODS:
    if cfg not in FINAL: continue
    S = sorted(FINAL[cfg])
    lm_ = [at(cfg, s, MID_READS)[0] for s in S]; lf_ = [at(cfg, s, FINAL_READS)[0] for s in S]
    af_ = [at(cfg, s, FINAL_READS)[1] for s in S]
    lines.append(f"| {LABEL[cfg]} | {BEST_ETA[cfg]:.0e} | {np.mean(lm_):.4f} ± {np.std(lm_):.4f} | "
                 f"{np.mean(lf_):.4f} ± {np.std(lf_):.4f} | {np.mean(af_):.3f} ± {np.std(af_):.3f} |")
lines += ["", "![SST-2](figures/colormezo_llm_sst2.svg)", "",
          f"*(moyenne ± écart-type sur {len(CFG['seeds'])} graines ; indicatif, pas de test statistique. À compléter avec la lecture des prédictions P6a–P6c ci-dessus.)*"]
block = "\n".join(lines)
open("readme_bloc_campagne_llm.md", "w").write(block)
print(block)
print("\nFichiers produits : colormezo_llm_sst2.png/.svg · readme_bloc_campagne_llm.md · ", RESULTS_PATH)
sauver()